# Setup

In [ ]:
import pandas as pd
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

import glob

import skimage as ski
from skimage import exposure, io, util, data, color, morphology, measure
from skimage.measure import regionprops
from skimage.transform import hough_circle, hough_circle_peaks
from skimage.feature import canny
from skimage.draw import circle_perimeter
from skimage.util import img_as_ubyte
from skimage.color import label2rgb
from skimage.filters import threshold_otsu, threshold_yen, threshold_local, gaussian, threshold_niblack, threshold_sauvola, sobel
from skimage.segmentation import watershed
from skimage.morphology import closing, footprint_rectangle, erosion, binary_opening, disk
from itertools import chain

import scipy as sp
from scipy import ndimage as ndi

import os
import json

In [ ]:
input  = '/content/drive/MyDrive/Vision/Prepared/1_coins/'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# dynamic figure shortcuts - v1.3

def P(a=False, title='', size=2, axis=False, cmap='inferno', interpolation='bilinear', bins=False, fontsize=12, dpi=75):
    global FIG
    if 'FIG' not in globals():
        # first subplot
        FIG = plt.figure(figsize=(size, size), dpi=dpi)
        ax = FIG.add_subplot(1, 1, 1)
    else:
        # change the geometry and add a new subplot
        n = len(FIG.axes)
        FIG.set_figwidth(FIG.get_figheight() * (n + 1))
        gs = FIG.add_gridspec(1, n + 1)
        for i in range(n):
            FIG.axes[i].set_subplotspec(gs[i])
        ax = FIG.add_subplot(gs[-1])

    ax.axis(axis)
    if title: ax.set_title(title)
    if type(a) == bool: pass
    elif type(a) == np.ndarray and bins != False:
        ax.hist(a.ravel(), bins=bins)
        ax.set_aspect(np.diff(ax.get_xlim())[0] / np.diff(ax.get_ylim())[0])
    elif type(a) == np.ndarray and a.ndim == 2: ax.imshow(a, cmap=cmap, interpolation=interpolation)
    elif type(a) == np.ndarray and a.ndim == 3: ax.imshow(a, interpolation=interpolation)
    elif type(a) == str: ax.text(0, 0, a, fontsize=fontsize, fontfamily='monospace')
    else: ax.plot(a)

def S():
    global FIG
    if 'FIG' in globals(): plt.show(); del FIG

def V(*args, **kwargs):
    P(*args, **kwargs); S()

# Loading

In [ ]:
df = pd.read_csv(input + 'labels.csv', sep=',')

In [ ]:
df.describe()

In [ ]:
files = glob.glob(input + '175*.png')
print(len(files))

In [ ]:
def get_coins(image_name) -> list[int]:
  image_name = os.path.splitext(os.path.basename(image_name))[0]

  # get labels
  labels_series = df.loc[df['name'] == image_name, 'labels']

  # for each label (string) separe by comma and remove space
  parts = list(chain.from_iterable(
      [str(s).strip().split(',') for s in labels_series.tolist()]
  ))

  return list(map(int, filter(lambda s: s.lower() != 'finger', parts)))

# Region based Method


In [ ]:
for file in files[25:]:
  image = io.imread(file)
  image = util.img_as_ubyte(image)
  labels = get_coins(file)
  coins_count = len(labels)

  # create markers
  markers = np.zeros_like(image)
  markers[image > 50] = 1
  markers[image < 75] = 2

  elevation_map = sobel(image)
  P(elevation_map, 'elevation map', size=8, cmap='gray')

  segmentation = watershed(elevation_map, markers)
  P(segmentation, 'segmentation', size=8, cmap='gray')

  segmentation_dilated = morphology.dilation(segmentation, morphology.disk(1))
  P(segmentation_dilated, 'segmentation dilated', size=8, cmap='gray')

  segmentation_filled = ndi.binary_fill_holes(segmentation_dilated - 1)
  P(segmentation_filled, 'segmentation filled', size=8, cmap='gray')

  segmentation_erosion = morphology.erosion(segmentation_filled, morphology.disk(18))
  P(segmentation_erosion, 'segmentation erosion', size=8, cmap='gray')

  labeled_coins, _ = ndi.label(segmentation_erosion)
  result = label2rgb(labeled_coins, image=image)

  props = regionprops(labeled_coins) # filter tiny regions
  min_area = 25
  found = sum(1 for r in props if r.area >= min_area)

  P(result, f"Coins: {coins_count} | Found: {found}", size=8)
  #print(f"Coins: {coins_count} | Fo")
  S()